# 06 — Semantic Similarity Analysis
This notebook demonstrates how we use Sentence Transformers (now optimized with ONNX Runtime) to detect risky clauses by comparing their semantic meaning against a library of known risky clauses.

**Production file:** `backend/similarity.py`

In [ ]:
import os
import numpy as np
import torch
from optimum.onnxruntime import ORTModelForFeatureExtraction
from transformers import AutoTokenizer

# Ensure we are in the correct directory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

MODEL_DIR = os.path.join("data", "onnx_model")
if not os.path.exists(MODEL_DIR):
    print("ONNX model not found. Run 'optimum-cli export onnx...' first.")
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = ORTModelForFeatureExtraction.from_pretrained(MODEL_DIR)
    print("ONNX model loaded successfully.")

## 1. Define Helper Functions

In [ ]:
def get_embedding(text: str) -> np.ndarray:
    inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    token_embeddings = outputs.last_hidden_state
    attention_mask = inputs['attention_mask']
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    embedding = (sum_embeddings / sum_mask)[0]
    return embedding.cpu().numpy()

def cos_similarity(v1: np.ndarray, v2: np.ndarray) -> float:
    dot_prod = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    if norm_v1 == 0 or norm_v2 == 0:
        return 0.0
    return float(dot_prod / (norm_v1 * norm_v2))

## 2. Compare Clauses

In [ ]:
reference = "The Company shall not be liable for any damages of any kind arising from this Agreement."
test_clause_1 = "We take no responsibility and will not pay for any harm caused by using our service."
test_clause_2 = "The client must pay the invoice within 30 days."

ref_emb = get_embedding(reference)
emb_1 = get_embedding(test_clause_1)
emb_2 = get_embedding(test_clause_2)

sim_1 = cos_similarity(ref_emb, emb_1)
sim_2 = cos_similarity(ref_emb, emb_2)

print(f"Reference: {reference}\n")
print(f"Clause 1: {test_clause_1}")
print(f"Similarity: {sim_1:.3f} (High semantic match despite different words)\n")
print(f"Clause 2: {test_clause_2}")
print(f"Similarity: {sim_2:.3f} (Low match, unrelated topic)")